# Newton-Ring Center Predictor (cx, cy)

Given a single 224×224 RGB frame with a visible Newton ring, predict the ring centroid `(cx, cy)`
— specifically, the vector **from the image center `(0.5, 0.5)` to `(cx, cy)`**, in normalized
image coordinates (`cx / W - 0.5`, `cy / H - 0.5`).

**Data**: HDF5 files `data/robot_*.hdf5` (same as `diff_pred.ipynb`)
- `frame_i_x` — `(224, 224, 3)` uint8 BGR frame
- `frame_i_y` — `(3,)` float32 motor position `[x, y, z]` in mm
- `frame_i_ring` — `(3,)` float32 `[cx, cy, area]` Newton-ring feature (NaN if no ring found)

**Approach**: reuse the exact `diff_pred.ipynb` blur+diff+ViT backbone, loading its **latest
trained weights** and **freezing** them entirely — only a new head is trained here. Each dataset
item is `(frame_with_ring, frame_without_ring, cx, cy)`: a frame where a ring was detected, paired
with a random ring-free frame from the same episode. The backbone sees
`blur(frame_with_ring) - blur(frame_without_ring)` (same diff-image convention as `diff_pred`), and
the head predicts the ring's center-relative position from the resulting CLS token.

**Validation**: `robot_8.hdf5` is held out entirely, exactly as in `diff_pred.ipynb`.


### Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

! pip install -q transformers h5py

In [ ]:
import h5py
import multiprocessing
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from transformers import ViTModel, ViTConfig

multiprocessing.set_start_method('fork', force=True)
warnings.filterwarnings('ignore')

# ── paths ──────────────────────────────────────────────────────────────────────
CKPT_DIR = Path('/content/drive/MyDrive/transfer_stacking_data/checkpoints_cx_cy_pred')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

DIFF_PRED_CKPT_DIR = Path('/content/drive/MyDrive/transfer_stacking_data/checkpoints_diff_pred')

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'device: {device}')

## Data
### Load episode positions

In [ ]:
HDF5_DIR   = Path('/content/drive/MyDrive/transfer_stacking_data/hdf5')
HDF5_FILES = sorted(HDF5_DIR.glob('robot_*.hdf5'))
assert HDF5_FILES, f'No robot_*.hdf5 files found in {HDF5_DIR}'
assert len(HDF5_FILES) >= 2, 'Need at least 2 hdf5 files to hold one out for validation'

IMG_SIZE = 224

# Hold out robot_8.hdf5 entirely as a validation episode — its frames are never sampled during
# training. Sorted filenames put it last among robot_1..robot_8, matching diff_pred.ipynb's split.
TRAIN_FILES = HDF5_FILES[:-1]
VAL_FILES   = HDF5_FILES[-1:]
assert VAL_FILES[0].name == 'robot_8.hdf5', f'Expected robot_8.hdf5 as the held-out file, got {VAL_FILES[0].name}'
print(f'Train files: {[p.name for p in TRAIN_FILES]}')
print(f'Val file   : {[p.name for p in VAL_FILES]}')


def load_episode_data(files):
    episode_data = []
    for path in files:
        with h5py.File(path, 'r') as f:
            n = sum(1 for k in f.keys() if k.endswith('_x'))
            xyz = torch.from_numpy(
                np.stack([f[f'frame_{i}_y'][:] for i in range(n)]).astype(np.float32)
            )  # (n, 3)
            ring = np.stack([
                f[f'frame_{i}_ring'][:] if f'frame_{i}_ring' in f else np.full(3, np.nan, dtype=np.float32)
                for i in range(n)
            ]).astype(np.float32)
            ring = torch.from_numpy(ring)  # (n, 3) [cx, cy, area], NaN where no ring was found

        # A "no ring found" frame has a well-defined area (0) but no meaningful centroid — keep
        # cx/cy as NaN and zero only the area column.
        ring_found = ~torch.isnan(ring).any(dim=1)
        ring[~ring_found, 2] = 0.0

        episode_data.append({'path': str(path), 'n': n, 'xyz': xyz, 'ring': ring, 'ring_found': ring_found})
        print(f'{path.name}: {n} frames  xyz range [{xyz.min():.1f}, {xyz.max():.1f}] mm  '
              f'ring found {ring_found.sum().item()}/{n}')
    return episode_data


print('\nTrain episodes:')
episode_data = load_episode_data(TRAIN_FILES)
print('\nVal episode (robot_8.hdf5):')
val_episode_data = load_episode_data(VAL_FILES)

print(f'\nTrain total : {sum(ep["n"] for ep in episode_data):,} frames')
print(f'Val total   : {sum(ep["n"] for ep in val_episode_data):,} frames')

### Dataset
Each item is `(frame_with_ring, frame_without_ring, target)`, sampled from the **same episode**:
- `frame_with_ring`: a frame where a Newton ring was detected (iterated over, so every ring frame
  is seen once per epoch)
- `frame_without_ring`: a random ring-free frame from the same episode (the "blank" reference the
  backbone diffs against, same convention as `diff_pred`'s `blur(frame_j) - blur(frame_i)`)
- `target`: `(2,)` = `(cx / IMG_SIZE - 0.5, cy / IMG_SIZE - 0.5)` — the vector from the image
  center to the ring centroid, in normalized `[-0.5, 0.5]` image coordinates. Already naturally
  scaled, so (unlike `diff_pred`'s `Δx, Δy, Δcx, Δcy, Δarea`) no additional z-scoring is applied.

Episodes with no ring-free frames, or no ring frames at all, are skipped entirely (can't form a pair).

In [ ]:
class RingCenterDataset(Dataset):
    """(frame_with_ring, frame_without_ring, target) triples from the same episode.

    target = (cx / IMG_SIZE - 0.5, cy / IMG_SIZE - 0.5): vector from the image center to the
    ring centroid, in normalized image coordinates.
    """

    def __init__(self, episode_data, img_size=IMG_SIZE):
        self.img_size = img_size
        self.episodes = []
        for ep in episode_data:
            with_idx    = torch.nonzero(ep['ring_found'], as_tuple=True)[0]
            without_idx = torch.nonzero(~ep['ring_found'], as_tuple=True)[0]
            if len(with_idx) == 0 or len(without_idx) == 0:
                print(f'{Path(ep["path"]).name}: skipped (needs both ring and ring-free frames, '
                      f'got {len(with_idx)} ring / {len(without_idx)} ring-free)')
                continue
            self.episodes.append({**ep, 'with_idx': with_idx, 'without_idx': without_idx})
        assert self.episodes, 'No episode has both ring and ring-free frames'

        self.ep_ends = []
        total = 0
        for ep in self.episodes:
            total += len(ep['with_idx'])
            self.ep_ends.append(total)
        self.total = total

    def __len__(self):
        return self.total

    def _ep_of(self, idx):
        for k, end in enumerate(self.ep_ends):
            if idx < end:
                return k
        return len(self.ep_ends) - 1

    def _read_frame(self, path, frame_idx):
        with h5py.File(path, 'r') as f:
            bgr = f[f'frame_{frame_idx}_x'][:]       # (H, W, 3) uint8 BGR
        rgb = bgr[:, :, ::-1].copy()
        return torch.from_numpy(rgb).permute(2, 0, 1)  # (3, H, W) uint8

    def __getitem__(self, idx):
        k        = self._ep_of(idx)
        ep       = self.episodes[k]
        prev_end = self.ep_ends[k - 1] if k > 0 else 0
        local_i  = idx - prev_end

        with_frame_idx    = ep['with_idx'][local_i].item()
        without_frame_idx  = ep['without_idx'][torch.randint(0, len(ep['without_idx']), (1,)).item()].item()

        frame_with    = self._read_frame(ep['path'], with_frame_idx)
        frame_without = self._read_frame(ep['path'], without_frame_idx)

        cx, cy, _ = ep['ring'][with_frame_idx]
        target = torch.tensor([cx / self.img_size - 0.5, cy / self.img_size - 0.5])
        return frame_with, frame_without, target


dataset     = RingCenterDataset(episode_data)
loader      = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, drop_last=True)
print(f'Train dataset size: {len(dataset):,} ring frames/epoch')

val_dataset = RingCenterDataset(val_episode_data)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, drop_last=False)
print(f'Val dataset size  : {len(val_dataset):,} ring frames')

## Model

**Backbone**: the exact same blur + diff + blank-ViT encoder as `diff_pred.ipynb` (`hidden_size=192`,
6 layers, 3 heads). Its weights are loaded from the **latest** `diff_pred` checkpoint in
`DIFF_PRED_CKPT_DIR` and then **frozen** (`requires_grad_(False)`, kept in `.eval()` mode) — only
the new head below is trained.

**Head** (fresh, trainable): CLS token → small MLP → `(2,)` `[cx_center_offset, cy_center_offset]`.

In [ ]:
EMB_DIM    = 192    # blank ViT hidden size == CLS token dim
BLUR_SIGMA = 2.0     # fixed gaussian blur sigma applied to each frame before subtraction
TARGET_DIM = 2        # [cx_center_offset, cy_center_offset]


def make_gaussian_kernel(sigma: float, device: str):
    """(1, 1, k, k) normalized 2D gaussian kernel, k = 2*round(3*sigma)+1."""
    if sigma <= 0:
        return None
    radius = max(1, int(round(3 * sigma)))
    coords = torch.arange(-radius, radius + 1, dtype=torch.float32, device=device)
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    kernel_2d = torch.outer(g, g)
    kernel_2d = kernel_2d / kernel_2d.sum()
    return kernel_2d.view(1, 1, kernel_2d.shape[0], kernel_2d.shape[1])


def gaussian_blur(x: torch.Tensor, kernel: torch.Tensor | None) -> torch.Tensor:
    """(B, C, H, W) -> (B, C, H, W), blurred per-channel (depthwise conv, same padding)."""
    if kernel is None:
        return x
    c = x.shape[1]
    k = kernel.repeat(c, 1, 1, 1).to(dtype=x.dtype, device=x.device)
    pad = k.shape[-1] // 2
    return F.conv2d(x, k, padding=pad, groups=c)


_blur_kernel = make_gaussian_kernel(BLUR_SIGMA, device)

vit_cfg = ViTConfig(
    num_channels=3, image_size=IMG_SIZE, patch_size=16,
    hidden_size=EMB_DIM, num_hidden_layers=6,
    num_attention_heads=3, intermediate_size=768,
)
vit = ViTModel(vit_cfg, add_pooling_layer=False).to(device)

diff_pred_ckpts = sorted(DIFF_PRED_CKPT_DIR.glob('diff_pred_epoch_*.pt'))
assert diff_pred_ckpts, f'No diff_pred checkpoints found in {DIFF_PRED_CKPT_DIR} — train diff_pred.ipynb first'
_backbone_ckpt = torch.load(diff_pred_ckpts[-1], map_location=device, weights_only=False)
vit.load_state_dict(_backbone_ckpt['vit'])
print(f'Loaded frozen ViT backbone from: {diff_pred_ckpts[-1].name}')

for p in vit.parameters():
    p.requires_grad_(False)
vit.eval()   # frozen backbone — stays in eval mode throughout

n_vit = sum(p.numel() for p in vit.parameters())
print(f'ViT params (frozen): {n_vit:,}')


def to_float(x):
    """(B, 3, H, W) uint8 -> (B, 3, H, W) float32 [0,1] on device."""
    return x.to(device, dtype=torch.float32).div_(255.0)


def extract_embedding(frame_with_float, frame_without_float):
    """(B, 3, 224, 224) float32 [0,1] x2 -> (B, EMB_DIM) CLS token of the frozen blank ViT run on
    the blurred difference image blur(frame_with_ring) - blur(frame_without_ring).
    """
    with torch.no_grad():
        bw  = gaussian_blur(frame_with_float, _blur_kernel)
        bwo = gaussian_blur(frame_without_float, _blur_kernel)
        diff = bw - bwo                                          # (B, 3, H, W), range ~[-1, 1]
        hidden = vit(diff, interpolate_pos_encoding=False).last_hidden_state
        return hidden[:, 0]                                      # (B, EMB_DIM) CLS token


class CenterHead(nn.Module):
    """CLS token -> small MLP -> (2,): [cx_center_offset, cy_center_offset]."""

    def __init__(self, emb_dim=EMB_DIM, target_dim=TARGET_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(emb_dim),
            nn.Linear(emb_dim, 256),
            nn.SiLU(),
            nn.Linear(256, target_dim),
        )

    def forward(self, emb):
        return self.net(emb)


head = CenterHead(emb_dim=EMB_DIM, target_dim=TARGET_DIM).to(device)
n_head = sum(p.numel() for p in head.parameters())
print(f'Head params (trainable): {n_head:,}')

with torch.no_grad():
    _fw   = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _fwo  = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    _emb  = extract_embedding(_fw, _fwo)
    _out  = head(_emb)
    print(f'Embedding shape : {_emb.shape}')   # expect (2, 192)
    print(f'Output shape    : {_out.shape}')   # expect (2, 2)

## Training
Only `head` is optimized — the ViT backbone is frozen (loaded from `diff_pred`).

In [ ]:
NUM_EPOCHS = 50

optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


def run_validation():
    head.eval()
    losses = []
    err_sum   = torch.zeros(2)
    err_count = 0
    with torch.no_grad():
        for frame_with, frame_without, target in val_loader:
            frame_with    = to_float(frame_with)
            frame_without = to_float(frame_without)
            target        = target.to(device)

            emb  = extract_embedding(frame_with, frame_without)
            pred = head(emb)
            loss = F.huber_loss(pred, target, delta=1.0)

            err_sum   += (pred - target).abs().sum(dim=0).cpu()
            err_count += target.shape[0]
            losses.append(loss.item())
    head.train()
    return float(np.mean(losses)), err_sum / max(err_count, 1)


head.train()

for epoch in range(NUM_EPOCHS):
    losses = []
    train_err_sum   = torch.zeros(2)
    train_err_count = 0

    for frame_with, frame_without, target in tqdm(loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}', leave=False):
        frame_with    = to_float(frame_with)
        frame_without = to_float(frame_without)
        target        = target.to(device)

        emb  = extract_embedding(frame_with, frame_without)
        pred = head(emb)                                       # (B, 2)
        loss = F.huber_loss(pred, target, delta=1.0)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), max_norm=5.0)
        optimizer.step()

        train_err_sum   += (pred.detach() - target).abs().sum(dim=0).cpu()
        train_err_count += target.shape[0]
        losses.append(loss.item())

    scheduler.step()
    val_loss, val_abs_err = run_validation()
    train_abs_err = train_err_sum / max(train_err_count, 1)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  '
          f'train_loss={np.mean(losses):.4f}  lr={scheduler.get_last_lr()[-1]:.2e}')
    print(f'  train|err|: cx={train_abs_err[0]:.4f}  cy={train_abs_err[1]:.4f}  (normalized)')
    print(f'  val_loss={val_loss:.4f}  val|err|: cx={val_abs_err[0]:.4f}  cy={val_abs_err[1]:.4f}  (normalized)')

    if (epoch + 1) % 10 == 0 or epoch + 1 == NUM_EPOCHS:
        torch.save({
            'epoch': epoch + 1,
            'head': head.state_dict(),
            'backbone_ckpt': diff_pred_ckpts[-1].name,
        }, CKPT_DIR / f'cx_cy_pred_epoch_{epoch+1:04d}.pt')

## Evaluation
### Load checkpoint

In [ ]:
ckpts = sorted(CKPT_DIR.glob('cx_cy_pred_epoch_*.pt'))
if ckpts:
    ckpt = torch.load(ckpts[-1], map_location=device, weights_only=False)
    head.load_state_dict(ckpt['head'])
    print(f'Loaded: {ckpts[-1].name}  (epoch {ckpt["epoch"]}, backbone={ckpt["backbone_ckpt"]})')
else:
    print('No checkpoints — run training first.')

head.eval()

### Per-axis error distribution (cx, cy)
Held-out validation file: `robot_8.hdf5`.

In [ ]:
all_preds, all_gt = [], []

with torch.no_grad():
    for frame_with, frame_without, target in tqdm(val_loader, desc='Evaluating'):
        frame_with    = to_float(frame_with.to(device))
        frame_without = to_float(frame_without.to(device))
        emb  = extract_embedding(frame_with, frame_without)
        pred = head(emb).cpu()
        all_preds.append(pred)
        all_gt.append(target)

all_preds = torch.cat(all_preds).numpy()   # (N, 2) normalized [-0.5, 0.5]
all_gt    = torch.cat(all_gt).numpy()

# convert to raw pixel cx, cy for interpretability
all_preds_px = (all_preds + 0.5) * IMG_SIZE
all_gt_px    = (all_gt + 0.5) * IMG_SIZE
errors_px    = all_preds_px - all_gt_px

AXIS_NAMES = ['cx', 'cy']
print(f'Per-axis error — held-out validation file robot_8.hdf5 (n={len(all_gt)}):')
for i, name in enumerate(AXIS_NAMES):
    mae  = np.abs(errors_px[:, i]).mean()
    rmse = np.sqrt((errors_px[:, i] ** 2).mean())
    print(f'  {name}: MAE={mae:.2f} px   RMSE={rmse:.2f} px')

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for i, name in enumerate(AXIS_NAMES):
    ax  = axes[i]
    gt  = all_gt_px[:, i]
    pr  = all_preds_px[:, i]
    lim_lo = min(gt.min(), pr.min()) - 5
    lim_hi = max(gt.max(), pr.max()) + 5
    ax.scatter(gt, pr, s=4, alpha=0.3, color='steelblue', rasterized=True)
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], 'r--', lw=1.2, label='ideal')
    ax.set_xlim(lim_lo, lim_hi); ax.set_ylim(lim_lo, lim_hi)
    ax.set_xlabel(f'GT {name} (px)'); ax.set_ylabel(f'Pred {name} (px)')
    ax.set_title(f'{name}  MAE={np.abs(errors_px[:, i]).mean():.2f} px')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Qualitative: predicted ring center vs. time, using a fixed ring-free reference frame
One cell on a **training** episode, one on the **held-out validation** episode (`robot_8.hdf5`) —
same fixed-reference comparison, so you can directly compare how well the ring position is tracked
in each.

In [ ]:
Q_EPISODE_DATA = episode_data   # TRAIN episode
EPISODE_IDX    = 0              # index into Q_EPISODE_DATA

ep = Q_EPISODE_DATA[EPISODE_IDX]
with_idx    = torch.nonzero(ep['ring_found'], as_tuple=True)[0]
without_idx = torch.nonzero(~ep['ring_found'], as_tuple=True)[0]
assert len(with_idx) > 0 and len(without_idx) > 0, 'Episode needs both ring and ring-free frames'
REF_FRAME_IDX = without_idx[0].item()   # fixed ring-free reference frame
steps = with_idx.tolist()


def read(idx):
    with h5py.File(ep['path'], 'r') as f:
        bgr = f[f'frame_{idx}_x'][:]
    rgb = bgr[:, :, ::-1].copy()
    return to_float(torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device))


all_preds_seq, all_gt_seq = [], []

with torch.no_grad():
    f_ref = read(REF_FRAME_IDX)
    for t in tqdm(steps, desc='Sequence eval'):
        ft   = read(t)
        emb  = extract_embedding(ft, f_ref)
        pred = head(emb).squeeze(0).cpu().numpy()
        cx, cy, _ = ep['ring'][t].numpy()
        gt = np.array([cx / IMG_SIZE - 0.5, cy / IMG_SIZE - 0.5])
        all_preds_seq.append(pred)
        all_gt_seq.append(gt)

all_preds_seq = (np.stack(all_preds_seq) + 0.5) * IMG_SIZE   # (N, 2) px
all_gt_seq    = (np.stack(all_gt_seq) + 0.5) * IMG_SIZE

print('Per-axis MAE — training file (fixed ring-free reference):')
for i, name in enumerate(AXIS_NAMES):
    mae = np.abs(all_preds_seq[:, i] - all_gt_seq[:, i]).mean()
    print(f'  {name}: {mae:.2f} px   (n={len(steps)})')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for i, name in enumerate(AXIS_NAMES):
    axes[i].plot(steps, all_gt_seq[:, i],   label='GT',   lw=1.2, marker='.', ms=3, linestyle='')
    axes[i].plot(steps, all_preds_seq[:, i], label='Pred', lw=1.2, marker='.', ms=3, linestyle='')
    axes[i].set_ylabel(f'{name} (px)')
    axes[i].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('frame index (ring-found frames only)')
plt.suptitle(f'[TRAIN] Ring center vs. ring-free reference frame {REF_FRAME_IDX} — episode {EPISODE_IDX}', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
Q_EPISODE_DATA = val_episode_data   # VALIDATION (held-out) episode: robot_8.hdf5
EPISODE_IDX    = 0                  # index into Q_EPISODE_DATA

ep = Q_EPISODE_DATA[EPISODE_IDX]
with_idx    = torch.nonzero(ep['ring_found'], as_tuple=True)[0]
without_idx = torch.nonzero(~ep['ring_found'], as_tuple=True)[0]
assert len(with_idx) > 0 and len(without_idx) > 0, 'Episode needs both ring and ring-free frames'
REF_FRAME_IDX = without_idx[0].item()   # fixed ring-free reference frame
steps = with_idx.tolist()


def read(idx):
    with h5py.File(ep['path'], 'r') as f:
        bgr = f[f'frame_{idx}_x'][:]
    rgb = bgr[:, :, ::-1].copy()
    return to_float(torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0).to(device))


all_preds_seq, all_gt_seq = [], []

with torch.no_grad():
    f_ref = read(REF_FRAME_IDX)
    for t in tqdm(steps, desc='Sequence eval'):
        ft   = read(t)
        emb  = extract_embedding(ft, f_ref)
        pred = head(emb).squeeze(0).cpu().numpy()
        cx, cy, _ = ep['ring'][t].numpy()
        gt = np.array([cx / IMG_SIZE - 0.5, cy / IMG_SIZE - 0.5])
        all_preds_seq.append(pred)
        all_gt_seq.append(gt)

all_preds_seq = (np.stack(all_preds_seq) + 0.5) * IMG_SIZE   # (N, 2) px
all_gt_seq    = (np.stack(all_gt_seq) + 0.5) * IMG_SIZE

print('Per-axis MAE — held-out validation file robot_8.hdf5 (fixed ring-free reference):')
for i, name in enumerate(AXIS_NAMES):
    mae = np.abs(all_preds_seq[:, i] - all_gt_seq[:, i]).mean()
    print(f'  {name}: {mae:.2f} px   (n={len(steps)})')

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for i, name in enumerate(AXIS_NAMES):
    axes[i].plot(steps, all_gt_seq[:, i],   label='GT',   lw=1.2, marker='.', ms=3, linestyle='')
    axes[i].plot(steps, all_preds_seq[:, i], label='Pred', lw=1.2, marker='.', ms=3, linestyle='')
    axes[i].set_ylabel(f'{name} (px)')
    axes[i].legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('frame index (ring-found frames only)')
plt.suptitle(f'[VAL] Ring center vs. ring-free reference frame {REF_FRAME_IDX} — held-out episode robot_8.hdf5', fontsize=11)
plt.tight_layout()
plt.show()